# Day 17 · Chunking: same data, different answers

**First-principles question:** How does the way you cut documents change what can be found?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import json
import re
import matplotlib.pyplot as plt
from lab.acme import load_kb
from lab.search import Index

kb = load_kb()
queries = [json.loads(l) for l in (ROOT / "data" / "acme" / "queries.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
print(len(kb), "articles ·", len(queries), "queries")

## 1 · Three ways to cut the articles
Every chunk is a dict: `{"id": unique, "article": "KB05", "text": ...}`.

In [ ]:
def by_article(articles):
    return [{"id": a["id"], "article": a["id"], "text": a["text"]} for a in articles]

def by_sentence(articles):
    # TODO: one chunk per sentence. Split a["text"] with re.split(r"(?<=[.!?])\s+", ...)
    #       Give each chunk an id like "KB05-2"
    ...

def by_fixed_size(articles, size=120):
    # TODO: one chunk per `size` characters of a["text"] (a["text"][i:i + size])
    ...

def with_title(chunker):
    """The 'contextual' trick: put the article title in front of every chunk."""
    titles = {a["id"]: a["title"] for a in kb}
    def chunk(articles):
        return [dict(c, text=f"{titles[c['article']]}: {c['text']}") for c in chunker(articles)]
    return chunk

STRATEGIES = {"article": by_article, "sentence": by_sentence, "fixed 120": by_fixed_size,
              "sentence + title": with_title(by_sentence)}
for name, chunker in STRATEGIES.items():
    print(f"{name:18} {len(chunker(kb)):4} chunks")

## 2 · recall@3 for each strategy
A hit = any chunk from the right article in the top 3.

In [ ]:
def recall_at_3(chunks):
    index = Index(chunks)
    hits = 0
    for q in queries:
        top = index.search(q["query"], k=3)
        hits += any(h["article"] == q["article"] for h in top)
    return hits / len(queries)

results = {name: (recall_at_3(chunker(kb)), len(chunker(kb))) for name, chunker in STRATEGIES.items()}
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(results.keys(), [r for r, _ in results.values()], color="#2153c4")
for i, (r, n) in enumerate(results.values()):
    ax.text(i, r + 0.02, f"{r:.0%} · {n} chunks", ha="center", fontsize=8)
ax.set_ylim(0, 1.1); ax.set_ylabel("recall@3"); plt.show()

**My strategy for Acme, and why:** …

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
one = [{"id": "X", "title": "T", "text": "First one. Second one! Third?"}]
assert len(by_sentence(one)) == 3, "by_sentence: 3 sentences → 3 chunks"
assert all(c["article"] == "X" for c in by_sentence(one))
assert len(by_fixed_size([{"id": "X", "title": "T", "text": "a" * 250}])) == 3, "250 characters → 3 chunks of 120"
assert len({c["id"] for c in by_sentence(kb)}) == len(by_sentence(kb)), "chunk ids must be unique"
assert len(results) >= 4
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")